In [ ]:
# Example 1: tokenize a trivial Python source string and inspect the first few tokens.
import pytokens

source = "x = 1 + 2\n"
tokens = list(pytokens.tokenize(source))

print(f"Got {len(tokens)} tokens for: {source!r}")
for tok in tokens[:8]:
    print(f"  {tok}")

assert len(tokens) > 0, "expected at least one token"


In [ ]:
# Example 2: each Token exposes a token type (kind) and source slice info.
import pytokens

source = "def foo():\n    return 42\n"
tokens = list(pytokens.tokenize(source))

# Inspect attributes on the first token to confirm the dataclass-ish shape.
first = tokens[0]
attrs = [a for a in dir(first) if not a.startswith('_')]
print(f"Token attrs: {attrs}")
print(f"First token repr: {first!r}")

assert hasattr(first, 'type') or hasattr(first, 'kind') or attrs, "token has no public attrs"


In [ ]:
# Example 3: tokenize a string literal and look for a STRING-like token.
import pytokens

source = 's = "hello"\n'
tokens = list(pytokens.tokenize(source))

# Render every token's stringified form so we can grep for the literal.
rendered = [repr(t) for t in tokens]
print("\n".join(rendered))

joined = " ".join(rendered).lower()
assert 'hello' in joined or 'string' in joined, "expected string literal in token stream"


In [ ]:
# Example 4: tokenize an indented block — verify the token count scales with content.
import pytokens

src_short = "a = 1\n"
src_long = "a = 1\nb = 2\nc = 3\nd = 4\ne = 5\n"

n_short = len(list(pytokens.tokenize(src_short)))
n_long = len(list(pytokens.tokenize(src_long)))

print(f"short ({len(src_short)} chars) -> {n_short} tokens")
print(f"long  ({len(src_long)} chars) -> {n_long} tokens")

assert n_long > n_short, "expected more tokens for longer source"


In [ ]:
# Example 5: An unterminated string literal is rejected.
import pytokens

# The string literal on this line is never closed.
bad = 's = "unterminated\n'

# pytokens raises UnterminatedString, a TokenizeError subclass, for an unterminated string.
try:
    tokens = list(pytokens.tokenize(bad))
except pytokens.UnterminatedString as e:
    assert isinstance(e, pytokens.TokenizeError), type(e).__mro__
    print(f"Rejected as expected ({type(e).__name__})")
else:
    raise AssertionError(f"tokenizer accepted an unterminated string: {tokens[:6]}")